[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/B.%20Core%20Quay-Side%20Operations%20%28The%20Ship-to-Shore%20Interface%29/15.%20The%20Vessel%20Re-stow%20Problem%20%28Transshipment%29/P15-Tier-3_executed.ipynb)

---


# 15. The Vessel Re-stow Problem (Transshipment)
## Tier 3: Ant Colony Optimization

This notebook uses Ant Colony Optimization (ACO) to assign each container to a vessel position. Each ant constructs a complete assignment, a pairwise-swap local search improves it, and pheromone trails guide later ants toward assignments that have worked well.

### Purpose, inputs, and outputs

- **Inputs:** Container identifiers, destinations, weights, current positions, discharge-sequence data, candidate vessel positions, and ACO parameters.
- **Output:** A feasible container-to-position assignment, its movement cost and move count, the best-cost history, and the final pheromone matrix.
- **Solution method:** Probabilistic ACO construction, pairwise assignment-swap local search, and iterative pheromone evaporation and deposition.

### Model assumptions and scope

Each container must be assigned to exactly one distinct candidate position. A position is feasible for a container only when it is accessible and its maximum weight is at least the container's weight. A move is counted when the assigned bay, row, or tier differs from the container's current coordinates. Every move has the same cost of $150.

The example represents positions as independent slots. It does not model stack interactions, crane travel, handling time, vessel stability, reefer connections, or operational precedence constraints. Destination ports and discharge sequences are included as container data, but they do not impose a segregation constraint or contribute directly to the objective. The heuristic uses bay proximity, weight feasibility, accessibility, and a preference for tier 1; it is guidance for constructing assignments, not an additional cost term.

### ACO concepts and mathematical model

A **pheromone** value records how strongly the search has learned to favor assigning a particular container to a particular position. A **heuristic value** provides immediate, problem-specific guidance before that learning has accumulated. Each ant uses both kinds of information to choose among feasible positions that have not yet been assigned in its solution.

Let $C$ be the set of containers and $P$ the set of candidate positions. For container $i \in C$ and position $j \in P$, let $\tau_{ij}$ be the pheromone level and $\eta_{ij}$ the heuristic value. At a construction step, let $A_i$ be the feasible positions still available to container $i$. The transition probability is

$$
p_{ij} = \frac{\tau_{ij}^{\alpha} \eta_{ij}^{\beta}}{\sum_{k \in A_i} \tau_{ik}^{\alpha} \eta_{ik}^{\beta}}, \qquad j \in A_i.
$$

Here, $\alpha$ controls the influence of learned pheromone, while $\beta$ controls the influence of the heuristic. With probability $q_0$, the ant selects a position with the largest transition probability; otherwise, it samples a position according to these probabilities. The code implements this rule in `calculate_transition_probabilities` and `construct_ant_solution`.

For an assignment $x$, let $m(x)$ be the number of containers whose assigned coordinates differ from their current coordinates. The objective and feasibility requirements are

$$
\begin{aligned}
\min_x \quad & f(x) = 150\,m(x), \\
\text{subject to} \quad
& \text{each container is assigned to exactly one position}, \\
& \text{each position is assigned to at most one container}, \\
& \text{each assigned position is accessible and can support the container's weight}.
\end{aligned}
$$

The cost function `calculate_solution_cost` evaluates this objective. The construction procedure enforces one-to-one assignment and filters infeasible positions; the local search preserves feasibility when it swaps assignments. The fixed movement charge means that minimizing cost is equivalent to minimizing the number of moved containers.

After each iteration, pheromone evaporates and assignments from that iteration deposit pheromone. In this implementation, an assignment with cost $f$ deposits an amount proportional to $1/(1+f)$, so even a zero-cost assignment can reinforce its choices. The best assignment of the iteration receives an additional deposit. With evaporation rate $\rho$, the update has the form

$$
\tau_{ij} \leftarrow (1-\rho)\tau_{ij} + \Delta_{ij},
$$

where $\Delta_{ij}$ is the total deposit on the container-position pair, including the extra deposit when that pair belongs to the iteration's best assignment. `update_pheromone` performs this update.

The pairwise-swap local search examines neighboring assignments formed by swapping the positions of two containers. It accepts a swap only when the resulting assignment is feasible and has lower cost. The function is named `local_search_2opt`; here, its neighborhood is specifically pairwise assignment swaps. If there are $n$ containers, $m$ positions, $A$ ants, $T$ ACO iterations, and at most $L$ local-search passes per ant, construction takes $O(nm)$ per ant and the cost-evaluating swap search takes at most $O(Ln^3)$ per ant. Thus, a rough upper bound for the main search is $O(AT(nm+Ln^3))$, in addition to pheromone updates. ACO is stochastic and does not guarantee a globally optimal assignment.

### How to interpret the results

The convergence history records the best cost found so far after each ACO iteration; it is therefore non-increasing. The pheromone matrix shows learned preferences, not proof that a position is operationally superior. Assignment tables and movement statistics describe the returned feasible assignment. Sensitivity results are observations from separate stochastic runs, not guarantees about parameter settings. No benchmark results or measured comparisons with other solution methods are supplied by this notebook.

In [ ]:
from typing import Dict, List, Tuple

import random
import time
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

In [ ]:
@dataclass
class Container:
    """Data for a container that may be assigned to a new vessel position."""

    id: str
    destination_port: str
    weight: float  # Tons.
    current_position: Tuple[int, int, int]  # (bay, row, tier).
    discharge_sequence: int  # Order in the discharge sequence.


@dataclass
class Position:
    """A candidate stow position and its individual feasibility attributes."""

    bay: int
    row: int
    tier: int
    accessible: bool = True
    max_weight: float = 50.0  # Tons per position.


@dataclass
class ACOConfig:
    """Parameters controlling the Ant Colony Optimization search."""

    num_ants: int = 20
    num_iterations: int = 100
    alpha: float = 1.0  # Pheromone importance.
    beta: float = 2.0  # Heuristic importance.
    rho: float = 0.1  # Pheromone evaporation rate.
    q0: float = 0.9  # Probability of choosing the most likely position.
    tau0: float = 1e-6  # Initial pheromone level.


@dataclass
class ACOResult:
    """Search results and diagnostics returned by ACO."""

    best_solution: Dict[str, Position]
    best_cost: float
    best_moves: int
    convergence_history: List[float]
    pheromone_matrix: np.ndarray
    computation_time: float

In [ ]:
def create_aco_problem():
    """Create the container and candidate-position data used in the example."""

    containers = [
        Container('HKG-1', 'HKG', 20.0, (1, 1, 1), 1),
        Container('SHA-1', 'SHA', 18.0, (1, 1, 2), 3),
        Container('HKG-2', 'HKG', 22.0, (2, 1, 1), 2),
        Container('SHA-2', 'SHA', 19.0, (2, 1, 2), 4),
        Container('HKG-3', 'HKG', 21.0, (3, 1, 1), 1),
        Container('SHA-3', 'SHA', 20.0, (3, 1, 2), 3),
        Container('HKG-4', 'HKG', 23.0, (4, 1, 1), 2),
        Container('SHA-4', 'SHA', 17.0, (4, 1, 2), 4),
        Container('HKG-5', 'HKG', 19.0, (5, 1, 1), 1),
        Container('SHA-5', 'SHA', 21.0, (5, 1, 2), 3),
        Container('HKG-6', 'HKG', 24.0, (6, 1, 1), 2),
        Container('SHA-6', 'SHA', 16.0, (6, 1, 2), 4),
    ]

    positions = [
        Position(bay=bay, row=1, tier=tier)
        for bay in range(1, 7)
        for tier in (1, 2)
    ]

    return containers, positions


containers, positions = create_aco_problem()
config = ACOConfig()

print(
    f'ACO problem created with {len(containers)} containers '
    f'and {len(positions)} positions.'
)
print(
    f'ACO configuration: {config.num_ants} ants, '
    f'{config.num_iterations} iterations.'
)

In [ ]:
def is_position_feasible(container: Container, position: Position) -> bool:
    """Check the accessibility and weight conditions for an assignment."""

    return position.accessible and container.weight <= position.max_weight


def calculate_heuristic_info(
    container: Container,
    position: Position,
) -> float:
    """Score a feasible container-position pair for ACO construction."""

    if not is_position_feasible(container, position):
        return 0.0

    bay_distance = abs(position.bay - container.current_position[0])
    heuristic = 1.0 / (1.0 + bay_distance)
    heuristic += 1.0
    heuristic += 0.5 if position.accessible else 0.0
    heuristic += 0.3 if position.tier == 1 else 0.0
    return heuristic


def initialize_pheromone_matrix(
    num_containers: int,
    num_positions: int,
    tau0: float,
) -> np.ndarray:
    """Give every container-position pair the same initial pheromone."""

    return np.full((num_containers, num_positions), tau0, dtype=float)


num_containers = len(containers)
num_positions = len(positions)

heuristic_matrix = np.zeros((num_containers, num_positions), dtype=float)
for container_idx, container in enumerate(containers):
    for position_idx, position in enumerate(positions):
        heuristic_matrix[container_idx, position_idx] = (
            calculate_heuristic_info(container, position)
        )

pheromone_matrix = initialize_pheromone_matrix(
    num_containers,
    num_positions,
    config.tau0,
)

print(f'Heuristic matrix shape: {heuristic_matrix.shape}')
print(f'Initial pheromone matrix shape: {pheromone_matrix.shape}')
print(f'Initial pheromone level: {config.tau0}')

In [ ]:
def calculate_transition_probabilities(
    available_positions: List[int],
    container_idx: int,
    pheromone: np.ndarray,
    heuristic: np.ndarray,
    alpha: float,
    beta: float,
) -> np.ndarray:
    """Normalize pheromone and heuristic scores over feasible choices."""

    if not available_positions:
        raise ValueError('At least one feasible position must be available.')

    scores = np.array(
        [
            pheromone[container_idx, position_idx] ** alpha
            * heuristic[container_idx, position_idx] ** beta
            for position_idx in available_positions
        ],
        dtype=float,
    )
    total_score = scores.sum()

    if total_score > 0.0:
        return scores / total_score

    return np.full(len(available_positions), 1.0 / len(available_positions))


def construct_ant_solution(
    containers: List[Container],
    positions: List[Position],
    pheromone: np.ndarray,
    heuristic: np.ndarray,
    config: ACOConfig,
) -> Dict[str, Position]:
    """Construct one complete, one-to-one feasible assignment."""

    solution = {}
    used_position_indices = set()
    container_order = list(range(len(containers)))
    random.shuffle(container_order)

    for container_idx in container_order:
        container = containers[container_idx]
        available_positions = [
            position_idx
            for position_idx, position in enumerate(positions)
            if position_idx not in used_position_indices
            and is_position_feasible(container, position)
        ]

        if not available_positions:
            raise ValueError(
                f'No feasible unused position is available for {container.id}.'
            )

        probabilities = calculate_transition_probabilities(
            available_positions,
            container_idx,
            pheromone,
            heuristic,
            config.alpha,
            config.beta,
        )

        if random.random() < config.q0:
            selected_choice_idx = int(np.argmax(probabilities))
            selected_position_idx = available_positions[selected_choice_idx]
        else:
            selected_position_idx = int(
                np.random.choice(available_positions, p=probabilities)
            )

        solution[container.id] = positions[selected_position_idx]
        used_position_indices.add(selected_position_idx)

    return solution


def calculate_solution_cost(
    solution: Dict[str, Position],
    containers: List[Container],
) -> Tuple[float, int]:
    """Return the fixed-charge movement cost and number of moved containers."""

    container_by_id = {container.id: container for container in containers}
    if len(container_by_id) != len(containers):
        raise ValueError('Container identifiers must be unique.')
    if set(solution) != set(container_by_id):
        raise ValueError('A solution must assign every container exactly once.')

    assigned_coordinates = [
        (position.bay, position.row, position.tier)
        for position in solution.values()
    ]
    if len(set(assigned_coordinates)) != len(assigned_coordinates):
        raise ValueError('A position cannot be assigned to multiple containers.')

    total_moves = 0
    for container_id, position in solution.items():
        container = container_by_id[container_id]
        if not is_position_feasible(container, position):
            raise ValueError(
                f'Assigned position is infeasible for {container_id}.'
            )

        new_coordinates = (position.bay, position.row, position.tier)
        if new_coordinates != container.current_position:
            total_moves += 1

    total_cost = total_moves * 150.0
    return total_cost, total_moves

In [ ]:
def local_search_2opt(
    solution: Dict[str, Position],
    containers: List[Container],
    positions: List[Position],
) -> Dict[str, Position]:
    """Improve an assignment using feasible pairwise position swaps."""

    best_solution = solution.copy()
    best_cost, _ = calculate_solution_cost(best_solution, containers)
    container_by_id = {container.id: container for container in containers}
    max_iterations = 10

    for _ in range(max_iterations):
        improved = False
        container_ids = list(best_solution)

        for first_idx in range(len(container_ids)):
            for second_idx in range(first_idx + 1, len(container_ids)):
                first_id = container_ids[first_idx]
                second_id = container_ids[second_idx]
                first_position = best_solution[first_id]
                second_position = best_solution[second_id]

                if not is_position_feasible(
                    container_by_id[first_id], second_position
                ):
                    continue
                if not is_position_feasible(
                    container_by_id[second_id], first_position
                ):
                    continue

                neighbor_solution = best_solution.copy()
                neighbor_solution[first_id] = second_position
                neighbor_solution[second_id] = first_position
                neighbor_cost, _ = calculate_solution_cost(
                    neighbor_solution,
                    containers,
                )

                if neighbor_cost < best_cost:
                    best_solution = neighbor_solution
                    best_cost = neighbor_cost
                    improved = True

        if not improved:
            break

    return best_solution


def update_pheromone(
    pheromone: np.ndarray,
    solutions: List[Dict[str, Position]],
    costs: List[float],
    best_cost: float,
    rho: float,
    containers: List[Container],
    positions: List[Position],
) -> np.ndarray:
    """Evaporate trails, then reward assignments from the current ant cohort."""

    if not solutions or len(solutions) != len(costs):
        raise ValueError('Solutions and costs must be non-empty and aligned.')

    pheromone *= 1.0 - rho
    container_index = {
        container.id: idx for idx, container in enumerate(containers)
    }
    position_index = {
        (position.bay, position.row, position.tier): idx
        for idx, position in enumerate(positions)
    }

    for solution, cost in zip(solutions, costs):
        deposit = 1.0 / (1.0 + cost)
        for container_id, position in solution.items():
            coordinates = (position.bay, position.row, position.tier)
            pheromone[
                container_index[container_id],
                position_index[coordinates],
            ] += deposit

    best_idx = int(np.argmin(costs))
    iteration_best_cost = costs[best_idx]
    if not np.isclose(iteration_best_cost, best_cost):
        raise ValueError('best_cost must be the best cost in this iteration.')

    best_deposit = 2.0 / (1.0 + iteration_best_cost)
    for container_id, position in solutions[best_idx].items():
        coordinates = (position.bay, position.row, position.tier)
        pheromone[
            container_index[container_id],
            position_index[coordinates],
        ] += best_deposit

    return pheromone

In [ ]:
def ant_colony_optimization(
    containers: List[Container],
    positions: List[Position],
    config: ACOConfig,
) -> ACOResult:
    """Run ACO construction, local search, and pheromone updates."""

    start_time = time.perf_counter()

    if not containers:
        raise ValueError('At least one container is required.')
    if len(positions) < len(containers):
        raise ValueError('There must be at least one position per container.')
    if len({container.id for container in containers}) != len(containers):
        raise ValueError('Container identifiers must be unique.')
    if config.num_ants <= 0 or config.num_iterations <= 0:
        raise ValueError('The ant count and iteration count must be positive.')
    if not 0.0 <= config.rho <= 1.0:
        raise ValueError('The evaporation rate rho must be between 0 and 1.')
    if not 0.0 <= config.q0 <= 1.0:
        raise ValueError('The choice probability q0 must be between 0 and 1.')
    if config.alpha < 0.0 or config.beta < 0.0 or config.tau0 <= 0.0:
        raise ValueError('alpha and beta must be non-negative and tau0 positive.')

    position_coordinates = [
        (position.bay, position.row, position.tier)
        for position in positions
    ]
    if len(set(position_coordinates)) != len(position_coordinates):
        raise ValueError('Candidate position coordinates must be unique.')

    num_containers = len(containers)
    num_positions = len(positions)
    heuristic = np.zeros((num_containers, num_positions), dtype=float)
    for container_idx, container in enumerate(containers):
        for position_idx, position in enumerate(positions):
            heuristic[container_idx, position_idx] = (
                calculate_heuristic_info(container, position)
            )
        if not np.any(heuristic[container_idx] > 0.0):
            raise ValueError(
                f'No feasible candidate position exists for {container.id}.'
            )

    pheromone = initialize_pheromone_matrix(
        num_containers,
        num_positions,
        config.tau0,
    )
    best_solution = None
    best_cost = float('inf')
    best_moves = 0
    convergence_history = []

    for iteration in range(config.num_iterations):
        solutions = []
        costs = []

        for _ in range(config.num_ants):
            solution = construct_ant_solution(
                containers,
                positions,
                pheromone,
                heuristic,
                config,
            )
            solution = local_search_2opt(solution, containers, positions)
            cost, moves = calculate_solution_cost(solution, containers)

            solutions.append(solution)
            costs.append(cost)

            if cost < best_cost:
                best_solution = solution.copy()
                best_cost = cost
                best_moves = moves

        iteration_best_cost = min(costs)
        pheromone = update_pheromone(
            pheromone,
            solutions,
            costs,
            iteration_best_cost,
            config.rho,
            containers,
            positions,
        )
        convergence_history.append(best_cost)

        if (iteration + 1) % 10 == 0:
            print(
                f'Iteration {iteration + 1:3d}: '
                f'best cost = ${best_cost:7.2f}, '
                f'moves = {best_moves:2d}'
            )

    computation_time = time.perf_counter() - start_time
    return ACOResult(
        best_solution=best_solution,
        best_cost=best_cost,
        best_moves=best_moves,
        convergence_history=convergence_history,
        pheromone_matrix=pheromone,
        computation_time=computation_time,
    )


print('\n=== Running Ant Colony Optimization ===')
aco_result = ant_colony_optimization(containers, positions, config)

print('\n=== ACO Results ===')
print(f'Computation time: {aco_result.computation_time:.2f} seconds')
print(f'Best cost: ${aco_result.best_cost:.2f}')
print(f'Best moves: {aco_result.best_moves}')
print(f'Iterations recorded: {len(aco_result.convergence_history)}')

checked_cost, checked_moves = calculate_solution_cost(
    aco_result.best_solution,
    containers,
)
assert np.isclose(checked_cost, aco_result.best_cost)
assert checked_moves == aco_result.best_moves
assert len(aco_result.best_solution) == len(containers)
assert np.all(np.diff(aco_result.convergence_history) <= 0.0)

In [ ]:
def analyze_aco_solution(
    result: ACOResult,
    containers: List[Container],
) -> pd.DataFrame:
    """Display assignment details and summarize movements by destination."""

    assignments = []
    for container in containers:
        position = result.best_solution[container.id]
        destination_coordinates = (position.bay, position.row, position.tier)
        is_move = destination_coordinates != container.current_position
        assignments.append(
            {
                'container': container.id,
                'destination': container.destination_port,
                'from_position': container.current_position,
                'to_position': destination_coordinates,
                'is_move': is_move,
                'weight_tons': container.weight,
                'bay_displacement': abs(
                    position.bay - container.current_position[0]
                ),
            }
        )

    assignment_df = pd.DataFrame(assignments)
    print('\n=== Detailed Assignment Analysis ===')
    print(
        assignment_df[
            ['container', 'destination', 'from_position', 'to_position', 'is_move']
        ].to_string(index=False)
    )

    print('\n=== Movement Summary by Destination ===')
    for port in assignment_df['destination'].unique():
        port_assignments = assignment_df[
            assignment_df['destination'] == port
        ]
        bays_used = sorted(
            {coordinates[0] for coordinates in port_assignments['to_position']}
        )
        print(f'{port}:')
        print(f'  Assigned containers: {len(port_assignments)}')
        print(f'  Moved containers: {int(port_assignments["is_move"].sum())}')
        print(f'  Bays used: {bays_used}')

    return assignment_df


assignment_df = analyze_aco_solution(aco_result, containers)

In [ ]:
def visualize_aco_results(
    result: ACOResult,
    assignment_df: pd.DataFrame,
) -> None:
    """Plot convergence, assignments, movements, and pheromone diagnostics."""

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    ax1 = axes[0, 0]
    iterations = np.arange(1, len(result.convergence_history) + 1)
    ax1.plot(iterations, result.convergence_history, color='blue', linewidth=2)
    ax1.set_title('Best-Cost Convergence')
    ax1.set_xlabel('ACO iteration')
    ax1.set_ylabel('Best cost ($)')
    ax1.grid(True, alpha=0.3)

    ax2 = axes[0, 1]
    image = ax2.imshow(
        result.pheromone_matrix,
        cmap='YlOrRd',
        aspect='auto',
    )
    ax2.set_title('Final Pheromone Matrix')
    ax2.set_xlabel('Position index')
    ax2.set_ylabel('Container index')
    fig.colorbar(image, ax=ax2, label='Pheromone level')

    ax3 = axes[0, 2]
    bay_table = assignment_df.assign(
        bay=assignment_df['to_position'].map(lambda coordinates: coordinates[0])
    ).groupby(['bay', 'destination']).size().unstack(fill_value=0)
    bay_table.plot(kind='bar', stacked=True, ax=ax3)
    ax3.set_title('Assigned Containers by Bay and Destination')
    ax3.set_xlabel('Bay number')
    ax3.set_ylabel('Number of containers')
    ax3.legend(title='Destination port')
    ax3.tick_params(axis='x', rotation=0)

    ax4 = axes[1, 0]
    moved_count = int(assignment_df['is_move'].sum())
    unchanged_count = len(assignment_df) - moved_count
    ax4.pie(
        [moved_count, unchanged_count],
        labels=['Moved', 'Unchanged'],
        colors=['orange', 'lightgreen'],
        autopct='%1.1f%%',
        startangle=90,
    )
    ax4.set_title('Container Movement Status')

    ax5 = axes[1, 1]
    displacement_counts = assignment_df['bay_displacement'].value_counts().sort_index()
    ax5.bar(
        displacement_counts.index,
        displacement_counts.values,
        color='teal',
        alpha=0.8,
    )
    ax5.set_title('Bay Displacement in the Returned Assignment')
    ax5.set_xlabel('Absolute bay displacement (bays)')
    ax5.set_ylabel('Number of containers')

    ax6 = axes[1, 2]
    pheromone_values = result.pheromone_matrix.ravel()
    ax6.hist(
        pheromone_values,
        bins=20,
        color='purple',
        alpha=0.7,
        edgecolor='black',
    )
    ax6.set_title('Final Pheromone Distribution')
    ax6.set_xlabel('Pheromone level')
    ax6.set_ylabel('Number of container-position pairs')
    ax6.axvline(
        np.mean(pheromone_values),
        color='red',
        linestyle='--',
        label=f'Mean: {np.mean(pheromone_values):.2e}',
    )
    ax6.legend()

    fig.suptitle('Ant Colony Optimization: Assignment and Search Diagnostics')
    fig.tight_layout()
    plt.show()


visualize_aco_results(aco_result, assignment_df)

In [ ]:
def test_aco_parameters():
    """Run the defined parameter settings and retain each observed history."""

    test_configs = [
        ACOConfig(num_ants=10, num_iterations=50, alpha=1.0, beta=2.0, rho=0.1),
        ACOConfig(num_ants=20, num_iterations=100, alpha=1.0, beta=2.0, rho=0.1),
        ACOConfig(num_ants=30, num_iterations=150, alpha=1.0, beta=2.0, rho=0.1),
        ACOConfig(num_ants=20, num_iterations=100, alpha=0.5, beta=3.0, rho=0.1),
        ACOConfig(num_ants=20, num_iterations=100, alpha=2.0, beta=1.0, rho=0.1),
        ACOConfig(num_ants=20, num_iterations=100, alpha=1.0, beta=2.0, rho=0.2),
    ]
    config_names = [
        'Base',
        'More Ants',
        'More Iterations',
        'Less Pheromone, More Heuristic',
        'More Pheromone, Less Heuristic',
        'More Evaporation',
    ]
    results = []

    print('\n=== ACO Parameter Sensitivity Analysis ===')
    for config, name in zip(test_configs, config_names):
        print(f'\nTesting {name} configuration...')
        result = ant_colony_optimization(containers, positions, config)
        results.append(
            {
                'config': name,
                'cost': result.best_cost,
                'moves': result.best_moves,
                'time': result.computation_time,
                'iterations': len(result.convergence_history),
                'convergence_history': result.convergence_history,
            }
        )
        print(
            f'  Cost: ${result.best_cost:.2f}, '
            f'moves: {result.best_moves}, '
            f'time: {result.computation_time:.2f} seconds'
        )

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    axes[0, 0].bar(
        config_names,
        [result['cost'] for result in results],
        color='skyblue',
    )
    axes[0, 0].set_title('Best Cost by Configuration')
    axes[0, 0].set_ylabel('Best cost ($)')
    axes[0, 0].tick_params(axis='x', rotation=45)

    axes[0, 1].bar(
        config_names,
        [result['moves'] for result in results],
        color='lightcoral',
    )
    axes[0, 1].set_title('Best Move Count by Configuration')
    axes[0, 1].set_ylabel('Number of moves')
    axes[0, 1].tick_params(axis='x', rotation=45)

    axes[1, 0].bar(
        config_names,
        [result['time'] for result in results],
        color='lightgreen',
    )
    axes[1, 0].set_title('Computation Time by Configuration')
    axes[1, 0].set_ylabel('Computation time (seconds)')
    axes[1, 0].tick_params(axis='x', rotation=45)

    for result in results:
        history = result['convergence_history']
        axes[1, 1].plot(
            np.arange(1, len(history) + 1),
            history,
            label=result['config'],
        )
    axes[1, 1].set_title('Observed Best-Cost Histories')
    axes[1, 1].set_xlabel('ACO iteration')
    axes[1, 1].set_ylabel('Best cost ($)')
    axes[1, 1].legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    axes[1, 1].grid(True, alpha=0.3)

    fig.suptitle('ACO Parameter Sensitivity Analysis')
    fig.tight_layout()
    plt.show()

    return results


parameter_results = test_aco_parameters()

## Results and interpretation

The executed cells report the assignment, its movement cost and move count, the convergence history, and pheromone diagnostics for the run. These are observations from the stochastic search, not prespecified numerical outcomes. The convergence plot tracks the best-so-far objective; a flat segment means that no lower-cost assignment was recorded during those iterations, not that optimality has been proved.

The sensitivity experiment reruns the same ACO procedure with the listed parameter configurations. Its convergence panel uses the histories returned by those runs. Because the search uses random choices and no random seed is specified, repeated executions can produce different results. A fair comparison of configurations should account for this run-to-run variability rather than treating one run as a guarantee.

### What this model does and does not establish

The returned assignment satisfies the modeled one-to-one, accessibility, and weight requirements, and its reported objective is the fixed cost per moved container. Pheromone concentration indicates which container-position choices the search reinforced; it is not a measure of operational safety or a proof of global optimality. The model does not enforce destination-port clustering, despite reporting assignments by destination, and it does not represent vessel-wide stacking or handling constraints. Such operational requirements would need to be specified as additional model conditions before interpreting an assignment as an executable stowage plan.

ACO is useful here as a constructive, population-based search in which learned assignment preferences influence subsequent solutions. Its practical performance depends on the instance, parameter settings, and stochastic choices. The notebook provides no measured benchmark against another method, so solution quality and computation time should be interpreted only from the runs actually performed.